### Hypothesis Testing 
In this notebook we will utilize statistical learning methods to determine whether or not the input variables (X) are effective in explaining the predictor (Y). To do this, we must compute the standard errors of the coeficent for each input and T-test it to determine if the coeficent is statisticaly significant. 

In [2]:
# import data 
import pandas as pd 
from pathlib import Path 

# load directory 
dir = Path.cwd()

# point to the data opath file 
X_train = dir.parent / 'Data' / 'X_train.csv'
Y_train = dir.parent / 'Data'/ 'Y_train.csv'

# load data 
X_train = pd.read_csv(X_train)
Y_train = pd.read_csv(Y_train)

# make data 1D (data will load with index if not ignored)
Y_train = Y_train['Survived'].values.ravel() 


# K-Nearest Neighbors
Important Insight: Since KNN does not have p-values because it is a non-parametric model, we will use sequential feature selection which automatically handles the backward elimintation and determines the best input features by checking accuaracy/precision at each step. 


In [3]:
from sklearn.feature_selection import SequentialFeatureSelector # elimination process
from sklearn.neighbors import KNeighborsClassifier # model 

# initialze model with the gridsearch adjusted parameters 
knn = KNeighborsClassifier(leaf_size=10, weights='distance')

# backward elimination for KKN 
sfs = SequentialFeatureSelector(knn, 
                                n_features_to_select = 'auto', 
                                direction = 'backward', 
                                scoring = 'precision')
# fit data onto model 
sfs.fit(X_train, Y_train)

# best features
selected_features = X_train.columns[sfs.get_support()]

# results
selected_features

c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Index(['Pclass', 'Age', 'Parch', 'Fare', 'Embarked_C', 'interaction4',
       'interaction5', 'interaction7', 'Ratio4'],
      dtype='object')

# Logit Testing 
Results: Redundancy is prevelant between the interaction variables, causing the model to suffer when predicting the raw column fare which is made up of the engieneered features. Also, will be removing the redundant categories like male and embarked to allow the model to use one category as the baseline. 


In [4]:
import statsmodels.api as sm 

#convert bool objects to floats (statsmodel has a hard time seeing them as bool objects)
X_train_numeric = X_train.astype(float)
                    
# manually add a constant intercept to the data 
X_train_const = sm.add_constant(X_train_numeric)

# fit the model
model = sm.Logit(Y_train, X_train_const)
result = model.fit(maxiter = 1000)

# view results 
print(f'Summary {result.summary()}')

         Current function value: 0.434132
         Iterations: 1000
Summary                            Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  623
Model:                          Logit   Df Residuals:                      605
Method:                           MLE   Df Model:                           17
Date:                Tue, 21 Apr 2026   Pseudo R-squ.:                  0.3416
Time:                        19:43:36   Log-Likelihood:                -270.46
converged:                      False   LL-Null:                       -410.79
Covariance Type:            nonrobust   LLR p-value:                 1.090e-49
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.6335      0.610     -1.038      0.299      -1.829       0.562
Unnamed: 0       -0.0003      0.001     -0.514

c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


# Variance Inflation Factor
Important Insight: 3 of the input variables posted infinite outputs, flagging mathematical copies of each other. 1 input variable is right on the edge of being to high, meaning it's mostly redundant. These inputs will be futher removed from the dataset and hypthesis training will be conducted again to view any updated changes to the models performance. 

In [5]:
# variance inflation error
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ensure X has a constant intercept 
X = X_train_const.copy()

# calculate VIFs
vifs = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# create table 
pd.DataFrame({'feature' : X.columns, 'VIF': vifs})

c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
0,const,33.972990
1,Unnamed: 0,1.028472
2,Pclass,2.713332
3,Age,1.312006
4,SibSp,1.845261
5,Parch,1.626830
6,Fare,inf
7,Embarked_644,1.036426
8,Embarked_C,1.144463
9,Embarked_Q,1.137675


# Logit Final Testing 
Result: The Model has converged and after the removal of all non-statisically significant features, the results are great!  



In [6]:
# remove redundant columns 
X_train_const = X_train_const.drop(columns = ['Fare', 'interaction5', 'interaction6', 'interaction7', 
                                              'Embarked_644', 'Embarked_C', 'Unnamed: 0', 'Ratio4', 'Ratio5', 'Ratio1', 'Parch', 'Embarked_Q'])

# begin model fitting 
model = sm.Logit(Y_train, X_train_const)
results = model.fit(maxiter = 2000)

# results
print(f'Summary {results.summary()}')

Optimization terminated successfully.
         Current function value: 0.448785
         Iterations 6
Summary                            Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  623
Model:                          Logit   Df Residuals:                      616
Method:                           MLE   Df Model:                            6
Date:                Tue, 21 Apr 2026   Pseudo R-squ.:                  0.3194
Time:                        19:43:36   Log-Likelihood:                -279.59
converged:                       True   LL-Null:                       -410.79
Covariance Type:            nonrobust   LLR p-value:                 9.197e-54
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.1526      0.456      0.334      0.738      -0.742       1.047
Pclass      

# Linear and KNN Dataset creation

In [8]:
# drop features that will preventing the linear models from performing well 
Linear_df = X_train.drop(columns = ['Fare', 'interaction5', 'interaction6', 'interaction7', 
                                              'Embarked_644', 'Embarked_C', 'Unnamed: 0', 
                                              'Ratio4', 'Ratio5', 'Ratio1', 'Parch', 'Embarked_Q'])

# include the features that will increase peformance of KNN
KNN_df = X_train[['Pclass', 'Age', 'Parch', 'Fare', 'Embarked_C', 'interaction4',
       'interaction5', 'interaction7', 'Ratio4']]

# save datasets 
Linear_df.to_csv("Linear_df.csv")
KNN_df.to_csv("KNN_df.csv")